In [ ]:
# Read MV VSAM data in a day loop
from pathlib import Path
from obspy.core import UTCDateTime, read_inventory
from flovopy.processing.sam import VSAM
from week8.set_samba_data_root import DATA_ROOT

DEBUG = True # turns plotting on mostly 

# -----------------------------------------------------------------------------
# Load Response information for MVO stations (from SEISAN database)
# -----------------------------------------------------------------------------
RESPONSE_DIR = DATA_ROOT / 'SEISAN_DB' / 'CAL'
from obspy import read_inventory
stationxml = RESPONSE_DIR / 'MV.xml'
inv = read_inventory(stationxml)

# -----------------------------------------------------------------------------
# Directory for RSAM (SAM) data
# -----------------------------------------------------------------------------
# Expand "~" to your home directory and create the folder if it doesn't exist
SAM_DIR = Path('~').expanduser() / 'work' / 'sam_data'

# -----------------------------------------------------------------------------
# Define a source location for Soufriere Hills volcano. 
# Station distances to this lat/lon are used to "reduce" the displacement to 1 km distance.
# -----------------------------------------------------------------------------
source = {'lat':16.7164, 'lon':-62.1654}

import asl_env
from flovopy.asl.grid import Grid
from flovopy.asl.wrappers import run_single_event, find_event_files, run_all_events
from flovopy.asl.config import ASLConfig, tweak_config
gridobj = Grid.load(asl_env.GRIDFILE_DEFAULT)
DEBUG=False
baseline_cfg = ASLConfig(
    inventory=asl_env.INV,
    output_base=asl_env.OUTPUT_DIR,
    gridobj=gridobj,
    global_cache=asl_env.GLOBAL_CACHE,
    station_correction_dataframe=station_corrections_df,
    wave_kind="surface",
    speed=1.5,
    Q=23, 
    peakf=2.0,
    dist_mode="3d", 
    misfit_engine="r2",
    window_seconds=5.0,
    min_stations=5,
    sam_class=VSAM, 
    sam_metric="mean",
    debug=DEBUG,
)
baseline_cfg.build()

# -----------------------------------------------------------------------------
# Define time range for processing
# -----------------------------------------------------------------------------
startTime = UTCDateTime(2003, 7, 1)   # start date (inclusive)
endTime   = UTCDateTime(2003, 7, 14)  # end date (exclusive)
window_seconds = 60  # window length for RSAM/DSAM/VSEM in seconds (e.g., 60 for 1-minute metrics)

# Number of seconds in one day (used for stepping through time)
secondsPerDay = 60 * 60 * 24

# Total number of days (not strictly needed, but useful for reference/debugging)
numDays = (endTime - startTime) / secondsPerDay

# Initialize loop variable
daytime = startTime

# -----------------------------------------------------------------------------
# Loop over each day and compute RSAM
# -----------------------------------------------------------------------------
while daytime < endTime:

    # -------------------------------------------------------------------------
    # Step 1: Load waveform data from Seisan archive for one day
    # -------------------------------------------------------------------------
    print("=" * 80)
    print(f"Reading day: {daytime.date}")

    # -------------------------------------------------------------------------
    # Step 2: ASL
    # -------------------------------------------------------------------------
    result = run_single_event(
        mseed_file=event_files[0],
        cfg=baseline_cfg,
        refine_sector=False,
        station_gains_df=None,
        switch_event_ctag = True,
        topo_kw=topo_kw,
        mseed_units='m/s', # default units for miniseed files being used - probably "Counts" or "m/s"        
        reduce_time=True,
        debug=DEBUG,
    )

    # -------------------------------------------------------------------------
    # Step forward one day
    # -------------------------------------------------------------------------
    daytime += secondsPerDay

